# Household Waste Classification Using a Neural Network Implemented from Scratch

## Technical Report Companion Notebook

This notebook accompanies the technical report for the Foundations of Machine Learning final project (Track 2). It implements a multi-layer perceptron (MLP) from scratch using only NumPy, trains it on the Garbage Dataset (Kunwar, 2026), and demonstrates incremental improvements through architectural and hyperparameter tuning.

# Setup and Dataset download (utilising the kagglehub functions)

In [4]:
# Setup dependencies
import os
import warnings
warnings.filterwarnings('ignore')
import subprocess
import sys
import datetime
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# numpy setup for reprocebility 
np.random.seed(42)
print("NumPy version:", np.__version__)

NumPy version: 2.5.1


# Helperfunctions fir dataset download and userinteraction

In [ ]:
def ask_user_for_conf(promt: str) -> bool:
    while True:
        user:input = input(promt).strip().lower()
        if user_input == "y":
            return True
        elif user_input == "n":
            return False
        else:
            print("ivalid input. please use only use 'y' or 'n'.")

def check_and_install_requirements():
    """Check if kagglehub is installed, otherwise install from requirements.txt"""
    # In a notebook, we assume kagglehub may not be installed so we try to import it.
    try:
        import kagglehub
        print("kagglehub is already installed.")
        return True
    except ImportError:
        print("kagglehub not found. Attempting to install via pip...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub"])
            print("kagglehub installed successfully!")
            return True
        except subprocess.CalledProcessError as e:
            print(f"Failed to install kagglehub: {e}")
            return False

def check_dataset_exists(dataset_path: Path) -> bool:
    if not dataset_path.exists():
        return False
    try:
        files = list(dataset_path.glob("*"))
        return len(files) > 0
    except:
        return False

def is_dataset_recent(dataset_path: Path, max_age_days: int = 7) -> bool:
    if not dataset_path.exists():
        return False
    last_modified = datetime.datetime.fromtimestamp(dataset_path.stat().st_mtime)
    age = datetime.datetime.now() - last_modified
    return age.days < max_age_days

def download_dataset(dataset_name: str) -> str:
    """Download dataset using kagglehub, checking cache first."""
    try:
        import kagglehub
        cache_dir = Path.home() / ".cache" / "kagglehub" / "datasets"
        dataset_dir = cache_dir / dataset_name.replace("/", "-")
        
        if check_dataset_exists(dataset_dir) and is_dataset_recent(dataset_dir):
            print(f"Dataset '{dataset_name}' is already downloaded and recent (less than 7 days old).")
            print(f"   Path: {dataset_dir}")
            answer = ask_user_for_conf("Do you want to re-download anyway? (y/n): ")
            if answer:
                print("Re-downloading dataset...")
                path = kagglehub.dataset_download(dataset_name)
                print("Path to dataset files:", path)
                return path
            else:
                print("Using existing dataset.")
                return str(dataset_dir)
        else:
            if dataset_dir.exists() and not is_dataset_recent(dataset_dir):
                print(f"Dataset exists but is older than 7 days.")
            else:
                print(f"Dataset not found locally.")
            print("Downloading dataset...")
            path = kagglehub.dataset_download(dataset_name)
            print("Path to dataset files:", path)
            return path
    except ImportError:
        print("ERROR: kagglehub is not installed. Please install it first.")
        raise
    except Exception as e:
        print(f"Error checking dataset: {e}")
        print("Attempting fresh download...")
        try:
            import kagglehub
            path = kagglehub.dataset_download(dataset_name)
            print("Path to dataset files:", path)
            return path
        except Exception as e2:
            print(f"Failed to download dataset: {e2}")
            raise

# Install kagglehub if missing
if not check_and_install_requirements():
    print("Setup failed: Could not install kagglehub.")
    sys.exit(1)

# Download the dataset
dataset_name = "sumn2u/garbage-classification-v2"
downloaded_path = download_dataset(dataset_name)
print(f"\nDataset location: {downloaded_path}")

# The dataset contains three subfolders: 'original', 'standardized_256', 'standardized_384'.
# We use the 'original' images (they have the highest resolution and are not pre-scaled).
dataset_root = Path(downloaded_path) / "original" # change to "standardized_384" for stand. images (not sure yet what we want to use SN)
if not dataset_root.exists():
    # Fallback: maybe the images are directly in the downloaded path?
    print(f"Warning: 'original' subfolder not found at {dataset_root}. Trying root directory.")
    dataset_root = Path(downloaded_path)
    # Check if there are subdirectories that look like classes
    subdirs = [d for d in dataset_root.iterdir() if d.is_dir()]
    if len(subdirs) == 0:
        print("ERROR: No subdirectories found. Please check the dataset structure.")
        sys.exit(1)
else:
    print(f"Using 'original' subfolder as dataset root: {dataset_root}")

kagglehub is already installed.
Dataset not found locally.
Path to dataset files: /Users/svn_ngbr/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12

Dataset location: /Users/svn_ngbr/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12
Using 'original' subfolder as dataset root: /Users/svn_ngbr/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/12/original


Dataset Overview

The Garbage Dataset contains 13,348 images across 10 household waste categories:

Category	Count

Metal	    930
Glass	    1,736
Biological	699
Paper	    1,336
Battery	    756
Trash	    453
Cardboard	1,411
Shoes	    1,449
Clothes	    1,892
Plastic	    1,597

# Load and preprocess the dataset
We map original categories to German bin categories as per assignment:
    paper + cardboard -> paper
    shoes + clothes -> textiles (or residual)
    others stay as is: metal, glass, biological, battery, residual (trash), plastic


In [9]:
def load_garbage_dataset(data_root, target_size=(64, 64)):
    """
    Load images from the Garbage Dataset.
    Returns:
        X: flattened images (n_samples, 12288) for 64x64x3
        y: integer labels
        class_names: list of original class names
        german_labels: list of mapped german bin category names
    """
    data_root = Path(data_root)
    # Find all subdirectories (each is a class)
    class_dirs = [d for d in data_root.iterdir() if d.is_dir()]
    class_dirs = sorted(class_dirs)  # consistent ordering
    original_names = [d.name for d in class_dirs]
    
    # Mapping to German bin categories
    # disclaimer: The dataset may have slightly different names so we map based on keywords.
    mapping = {
        'paper': 'paper',
        'cardboard': 'paper',
        'shoes': 'textiles',
        'clothes': 'textiles',
        'metal': 'metal',
        'glass': 'glass',
        'biological': 'biological',
        'battery': 'battery',
        'trash': 'residual',
        'plastic': 'plastic'
    }
    # remove unwanted character
    german_labels = []
    for name in original_names:
        # Try to match by lowercasing and removing underscores/spaces
        key = name.lower().replace('_', '').replace(' ', '')
        # Find best match in mapping keys
        found = False
        for k, v in mapping.items():
            if k in key or key in k:
                german_labels.append(v)
                found = True
                break
        if not found:
            # keep original name
            german_labels.append(name)
    
    # create a unified label set
    unique_german = sorted(set(german_labels))
    # Remap to indices
    german_to_idx = {label: i for i, label in enumerate(unique_german)}
    y_german = [german_to_idx[label] for label in german_labels]
    
    X = []
    y = []
    class_counts = []
    
    print(f"Loading images from {len(class_dirs)} classes...")
    for class_idx, (class_dir, orig_name) in enumerate(zip(class_dirs, original_names)):
        image_files = [f for f in class_dir.iterdir() 
                       if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
        class_counts.append(len(image_files))
        print(f"  Class '{orig_name}' -> German: '{german_labels[class_idx]}' : {len(image_files)} images")
        
        for img_file in image_files:
            try:
                img = Image.open(img_file).convert('RGB')
                img = img.resize(target_size)
                img_array = np.array(img).astype(np.float32)
                X.append(img_array.flatten())
                y.append(german_to_idx[german_labels[class_idx]])
            except Exception as e:
                print(f"Warning: Could not load {img_file}: {e}")
                continue
    
    X = np.array(X)
    y = np.array(y)
    print(f"\nTotal images loaded: {X.shape[0]}, each with {X.shape[1]} features.")
    print(f"German bin categories: {unique_german} (mapped to indices {list(range(len(unique_german)))})")
    return X, y, unique_german

# Load data
X, y, german_categories = load_garbage_dataset(dataset_root, target_size=(64, 64))

Loading images from 10 classes...
  Class 'battery' -> German: 'battery' : 756 images
  Class 'biological' -> German: 'biological' : 699 images
  Class 'cardboard' -> German: 'paper' : 1411 images
  Class 'clothes' -> German: 'textiles' : 1892 images
  Class 'glass' -> German: 'glass' : 1736 images
  Class 'metal' -> German: 'metal' : 930 images
  Class 'paper' -> German: 'paper' : 1336 images
  Class 'plastic' -> German: 'plastic' : 1597 images
  Class 'shoes' -> German: 'textiles' : 1449 images
  Class 'trash' -> German: 'residual' : 453 images

Total images loaded: 12259, each with 12288 features.
German bin categories: ['battery', 'biological', 'glass', 'metal', 'paper', 'plastic', 'residual', 'textiles'] (mapped to indices [0, 1, 2, 3, 4, 5, 6, 7])
